# Python to C++/Rust Code Converter

This notebook converts Python code to either **C++** or **Rust** using OpenAI's GPT-OSS-120B model via OpenRouter.

**Setup for Google Colab:**
1. Add your OpenRouter API key to Colab Secrets: *Secrets* (key icon) → Add new secret → `OPENROUTER_API_KEY`
2. Run all cells below

## Google Colab Link:

https://colab.research.google.com/drive/1DvnBSweSE_BF9gdQcZnZsgYu9g1LgeOM?usp=sharing

In [ ]:
# Install dependencies (run once in Colab)
!pip install -q gradio openai

In [ ]:
# Imports
from openai import OpenAI
import gradio as gr

In [ ]:
# Connect to OpenRouter (API key from Colab Secrets)
from google.colab import userdata

OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
OPENROUTER_URL = "https://openrouter.ai/api/v1"
MODEL = "openai/gpt-oss-120b"

client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_URL)
print(f"Connected to OpenRouter. Model: {MODEL}")

In [ ]:
def convert_code(python_code: str, target_language: str) -> str:
    """Convert Python code to C++ or Rust using GPT-OSS-120B."""
    if not python_code or not python_code.strip():
        return "Please paste some Python code to convert."

    system_prompt = f"""Your task is to convert Python code into high-quality {target_language} code.
Respond ONLY with the {target_language} code. Do not include markdown code blocks, explanations, or any other text.
The output must be valid, runnable {target_language} code that preserves the logic and behavior of the Python code."""

    user_prompt = f"""Convert this Python code to {target_language}:

```python
{python_code}
```"""

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )
        code = response.choices[0].message.content
        # Strip markdown code blocks if the model includes them
        code = code.replace('```cpp', '').replace('```c++', '').replace('```rust', '').replace('```rs', '').replace('```', '').strip()
        return code
    except Exception as e:
        return f"Error during conversion: {str(e)}"

In [ ]:
# Build Gradio UI
with gr.Blocks(title="Python to C++/Rust Converter") as ui:
    gr.Markdown("## Convert Python Code to C++ or Rust")

    with gr.Row():
        with gr.Column():
            python_input = gr.Textbox(
                label="Paste your Python code here",
                placeholder="def hello():\n    print('Hello, World!')",
                lines=15,
                max_lines=30
            )
            target_language = gr.Radio(
                choices=["C++", "Rust"],
                value="C++",
                label="Convert to"
            )
            convert_btn = gr.Button("Convert")

        with gr.Column():
            output_code = gr.Textbox(
                label="Converted code",
                lines=15,
                max_lines=30,
                interactive=False
            )

    convert_btn.click(
        fn=convert_code,
        inputs=[python_input, target_language],
        outputs=output_code
    )

ui.launch(share=True)  # share=True makes it accessible in Colab